# Phase 6c Wave 3: Equivalence-Principle Classification of Phase 5x DM Mechanisms — Technical Notebook

Companion to `papers/paper34_equivalence_principle/paper_draft.tex`.

**Lean module:** `lean/SKEFTHawking/EquivalencePrinciple.lean` (24 substantive theorems = 12 original + 12 strengthening / 0 sorry / 0 new axioms; verified `propext, Classical.choice, Quot.sound` only via `lean_verify`).

**Structure (mirrors paper §1–§6):**
1. Setup: EP levels, mechanism enumeration, MICROSCOPE / STEP scales
2. The 6×3 EP-violation matrix (`EP_VIOLATION_MATRIX`)
3. Single-claim `violationLevel` per mechanism (12 strengthening theorems)
4. Subsumption monotonicity (`violatesAt_mono`)
5. Quantitative `norm_num` anchors (vestigial-phase η vs MICROSCOPE_BOUND, etc.)
6. Cross-tensions: `non_violators_share_violationLevel`, `fangGu_failure_mode_is_kinematic_not_ep`
7. Figure: 6×3 heatmap + η-scale log-comparison bar chart
8. Lean theorem inventory

All numerical content imports from `src.equivalence_principle` — no inline physics redefinition.

## 1. Setup: EP levels, mechanisms, experimental scales

**Equivalence-principle hierarchy** (see Will, *Theory and Experiment in Gravitational Physics*, 2nd ed., 2018):
- **WEP** (weak): all test masses fall identically.
- **EEP** (Einstein): WEP + local Lorentz invariance + position invariance for non-gravitational physics.
- **SEP** (strong): EEP + same statements applied to gravitational self-energy.

Implication strength: WEP ⊂ EEP ⊂ SEP. A WEP-violator automatically violates all three.

**Six Phase-5x dark-sector mechanisms** are being classified.
**Experimental anchors:** Touboul et al. (PRL 119, 231101, 2017) gives MICROSCOPE bound $\eta < 10^{-15}$; STEP-class projection $\eta \sim 10^{-18}$ (post-strengthening hedge in paper §II).

In [1]:
import os, sys
_HERE = os.getcwd()
_PROJ = _HERE if os.path.basename(_HERE) == 'SK_EFT_Hawking' else os.path.dirname(_HERE)
if _PROJ not in sys.path:
    sys.path.insert(0, _PROJ)

from src.equivalence_principle import (
    EPLevel,
    EPMechanism,
    MICROSCOPE_BOUND,
    STEP_TARGET,
    VESTIGIAL_PHASE_ETA_MAX,
    VESTIGIAL_RELICS_ETA,
    violation_level,
    violates_at,
    satisfies_at,
    EP_VIOLATION_MATRIX,
)
from src.core.visualizations import COLORS, fig_ep_violation_matrix

print('Levels (numerical_order):')
for L in EPLevel:
    print(f'  {L.name:5s}  order = {L.numerical_order}')

print()
print('Mechanisms (Phase 5x DM candidates):')
for m in EPMechanism:
    print(f'  {m.name}')

print()
print('Experimental / projected η scales:')
print(f'  MICROSCOPE bound          η < {MICROSCOPE_BOUND:.0e}      (Touboul 2017, current)')
print(f'  STEP-class target         η ~ {STEP_TARGET:.0e}      (projected design sensitivity)')
print(f'  vestigial-phase max       η_max = {VESTIGIAL_PHASE_ETA_MAX}      (full vestigial phase)')
print(f'  vestigial relics residual η ~ {VESTIGIAL_RELICS_ETA:.0e}      (defect remnants)')

Levels (numerical_order):
  WEP    order = 0
  EEP    order = 1
  SEP    order = 2

Mechanisms (Phase 5x DM candidates):
  vestigialDifferentialCoupling
  vestigialReliscSTEPClass
  fangGuTorsionTrace
  fractonSubdiffusion
  sfdmThomasFermi
  hiddenSectorZ16Singlet

Experimental / projected η scales:
  MICROSCOPE bound          η < 1e-15      (Touboul 2017, current)
  STEP-class target         η ~ 1e-18      (projected design sensitivity)
  vestigial-phase max       η_max = 1.0      (full vestigial phase)
  vestigial relics residual η ~ 1e-18      (defect remnants)


## 2. The 6×3 EP-violation matrix

Mirror of the Lean `violatesAt` predicate across all (mechanism, level) pairs. Two mechanisms violate WEP — both vestigial — and therefore violate EEP and SEP by subsumption. The other four satisfy all three levels.

In [2]:
header = f'{"mechanism":35s} | {"WEP":^9} | {"EEP":^9} | {"SEP":^9}'
print(header)
print('-' * len(header))
for m in EPMechanism:
    row = f'{m.name:35s}'
    for L in EPLevel:
        cell = 'violates' if EP_VIOLATION_MATRIX[m][L] else 'satisfies'
        row += f' | {cell:^9}'
    print(row)

print()
print('Cross-checks against the Lean violatesAt / satisfiesAt predicates:')
all_ok = True
for m in EPMechanism:
    for L in EPLevel:
        if EP_VIOLATION_MATRIX[m][L] != violates_at(m, L):
            all_ok = False
        if EP_VIOLATION_MATRIX[m][L] == satisfies_at(m, L):
            all_ok = False
print(f'  matrix consistent with predicates: {all_ok}')

mechanism                           |    WEP    |    EEP    |    SEP   
-----------------------------------------------------------------------
vestigialDifferentialCoupling       | violates  | violates  | violates 
vestigialReliscSTEPClass            | violates  | violates  | violates 
fangGuTorsionTrace                  | satisfies | satisfies | satisfies
fractonSubdiffusion                 | satisfies | satisfies | satisfies
sfdmThomasFermi                     | satisfies | satisfies | satisfies
hiddenSectorZ16Singlet              | satisfies | satisfies | satisfies

Cross-checks against the Lean violatesAt / satisfiesAt predicates:
  matrix consistent with predicates: True


## 3. Single-claim `violationLevel` per mechanism (strengthening pass)

The original wave shipped four 3-conjunct `*_satisfies_all_EP` bundles plus two `*_violates_WEP` per-mechanism theorems. The strengthening pass *added* single-claim `violationLevel m = some WEP` (violators) or `violationLevel m = none` (non-violators) forms alongside the originals (the bundles still exist), plus a shared `violatesAt_mono` extraction lemma — making the substantive content explicit while leaving downstream consumers of the 3-conjunct bundles intact.

In [3]:
for m in EPMechanism:
    vl = violation_level(m)
    if vl is None:
        marker = 'has_no_violation'
    else:
        marker = f'violates {vl.name}'
    print(f'  {m.name:35s} → violationLevel = {str(vl):20s}  Lean: *_{marker}')

  vestigialDifferentialCoupling       → violationLevel = EPLevel.WEP           Lean: *_violates WEP
  vestigialReliscSTEPClass            → violationLevel = EPLevel.WEP           Lean: *_violates WEP
  fangGuTorsionTrace                  → violationLevel = None                  Lean: *_has_no_violation
  fractonSubdiffusion                 → violationLevel = None                  Lean: *_has_no_violation
  sfdmThomasFermi                     → violationLevel = None                  Lean: *_has_no_violation
  hiddenSectorZ16Singlet              → violationLevel = None                  Lean: *_has_no_violation


## 4. Subsumption monotonicity (`violatesAt_mono`)

`violatesAt_mono`: if `epLevelOrder L₁ ≤ epLevelOrder L₂` and a mechanism violates at $L_1$, it violates at $L_2$. This is the structural lemma that derives the 3-conjunct `*_satisfies_all_EP` bundles from the single-claim `violationLevel` form, and ensures WEP-violators automatically violate EEP and SEP.

In [4]:
for m in [EPMechanism.vestigialDifferentialCoupling, EPMechanism.fangGuTorsionTrace]:
    print(f'  {m.name}:')
    for L1 in EPLevel:
        for L2 in EPLevel:
            if L1.numerical_order > L2.numerical_order:
                continue
            if violates_at(m, L1) and not violates_at(m, L2):
                print(f'    monotonicity violation at {L1.name} → {L2.name}')
    print(f'    (no monotonicity violations — matches Lean violatesAt_mono)')
    print()

  vestigialDifferentialCoupling:
    (no monotonicity violations — matches Lean violatesAt_mono)

  fangGuTorsionTrace:
    (no monotonicity violations — matches Lean violatesAt_mono)



## 5. Quantitative `norm_num` anchors

Two Lean theorems pin the qualitative classification to numerical content:

- `vestigial_phase_eta_violates_microscope_bound : (1.0 : ℝ) > 1.0e-15` — the maximal vestigial-phase $\eta$ exceeds MICROSCOPE by 15 orders.
- `vestigial_relics_below_microscope_bound : (1.0e-18 : ℝ) < 1.0e-15` — the relics residual $\eta$ is sub-MICROSCOPE (would require STEP-class detection).

(A self-equality tautology `step_target_can_test_vestigial_relics : 1.0e-18 ≤ 1.0e-18 := le_refl _` was *removed* in the post-wave strengthening audit; the structural STEP=relics equality is now read off by inspection from the constants module.)

In [5]:
import math

print('Vestigial-phase η = 1 vs MICROSCOPE bound:')
print(f'  {VESTIGIAL_PHASE_ETA_MAX} > {MICROSCOPE_BOUND}  →  {VESTIGIAL_PHASE_ETA_MAX > MICROSCOPE_BOUND}')
print(f'  ratio = {VESTIGIAL_PHASE_ETA_MAX / MICROSCOPE_BOUND:.0e}   ({math.log10(VESTIGIAL_PHASE_ETA_MAX / MICROSCOPE_BOUND):.0f} orders above bound — already EXCLUDED)')

print()
print('Vestigial relics η ~ 1e-18 vs MICROSCOPE bound:')
print(f'  {VESTIGIAL_RELICS_ETA} < {MICROSCOPE_BOUND}  →  {VESTIGIAL_RELICS_ETA < MICROSCOPE_BOUND}')
print(f'  ratio = {VESTIGIAL_RELICS_ETA / MICROSCOPE_BOUND:.0e}   ({-math.log10(MICROSCOPE_BOUND / VESTIGIAL_RELICS_ETA):.0f} orders below bound — STEP-detectable)')

print()
print('STEP-class target = vestigial relics residual:')
print(f'  STEP_TARGET = {STEP_TARGET}    VESTIGIAL_RELICS_ETA = {VESTIGIAL_RELICS_ETA}')
print(f'  match: {STEP_TARGET == VESTIGIAL_RELICS_ETA}   → STEP would saturate the predicted relic signature.')

Vestigial-phase η = 1 vs MICROSCOPE bound:
  1.0 > 1e-15  →  True
  ratio = 1e+15   (15 orders above bound — already EXCLUDED)

Vestigial relics η ~ 1e-18 vs MICROSCOPE bound:
  1e-18 < 1e-15  →  True
  ratio = 1e-03   (-3 orders below bound — STEP-detectable)

STEP-class target = vestigial relics residual:
  STEP_TARGET = 1e-18    VESTIGIAL_RELICS_ETA = 1e-18
  match: True   → STEP would saturate the predicted relic signature.


## 6. Cross-tensions and cross-bridges

**`non_violators_share_violationLevel`** — all four non-vestigial mechanisms (FangGu torsion, fracton subdiffusion, SFDM Thomas-Fermi, hidden-sector ℤ₁₆) share `violationLevel = none`. Consequence: they are *EP-degenerate* — EP precision experiments cannot distinguish them.

**`fangGu_failure_mode_is_kinematic_not_ep`** — the FangGu torsion mechanism's failure mode is kinematic (it cannot reproduce the observed CDM equation of state $w \approx 0$ because $w_{FG} = 1/3$ is fixed by the trace structure), NOT an EP violation. Imports `FangGuTorsionDM` and calls `fg_cdm_obstruction` directly — no docstring drift.

In [6]:
non_violators = [m for m in EPMechanism if violation_level(m) is None]
print(f'Non-violators ({len(non_violators)} mechanisms):')
for m in non_violators:
    print(f'  {m.name:35s} violation_level = None')
print(f'  → all share violationLevel = None  (Lean: non_violators_share_violationLevel)')
print(f'  → EP precision experiments cannot distinguish these four mechanisms.')

violators = [m for m in EPMechanism if violation_level(m) is not None]
print()
print(f'Violators ({len(violators)} mechanisms — both vestigial):')
for m in violators:
    print(f'  {m.name:35s} violation_level = {violation_level(m).name}')
print(f'  → ep_violation_is_vestigial_only structural conclusion.')

Non-violators (4 mechanisms):
  fangGuTorsionTrace                  violation_level = None
  fractonSubdiffusion                 violation_level = None
  sfdmThomasFermi                     violation_level = None
  hiddenSectorZ16Singlet              violation_level = None
  → all share violationLevel = None  (Lean: non_violators_share_violationLevel)
  → EP precision experiments cannot distinguish these four mechanisms.

Violators (2 mechanisms — both vestigial):
  vestigialDifferentialCoupling       violation_level = WEP
  vestigialReliscSTEPClass            violation_level = WEP
  → ep_violation_is_vestigial_only structural conclusion.


## 7. Figure — EP-violation matrix and η-scale comparison

**Left panel.** The 6×3 mechanism × EP-level matrix as a heatmap. Amber cells indicate violation; gray cells indicate satisfaction. The two-row amber band (vestigial mechanisms) sits above the four-row gray band (non-vestigial mechanisms) — the visual signature of `ep_violation_is_vestigial_only`.

**Right panel.** Four log-scale bars showing the η values that matter: vestigial-phase $\eta_{\max} = 1$ (already excluded by MICROSCOPE), MICROSCOPE bound $10^{-15}$, vestigial relics $\eta \sim 10^{-18}$, and STEP-class target $10^{-18}$. The horizontal dotted line at $\log_{10}(\text{MICROSCOPE\_BOUND}) = -15$ separates the excluded amber bars from the STEP-detectable blue bar.

In [7]:
# viz-ref: fig_ep_violation_matrix
fig_ep_violation_matrix().show()

## 8. Lean theorem inventory (24 substantive)

All 24 theorems in `EquivalencePrinciple.lean` are clean (zero `sorry`, zero new axioms; closure $\{\texttt{propext, Classical.choice, Quot.sound}\}$).

**Original 12 (Wave 6c.3):**
1–2. `vestigialDifferentialCoupling_violates_WEP`, `vestigialReliscSTEPClass_violates_WEP` — per-mechanism violators.
3–6. `*_satisfies_all_EP` — four non-violator bundles (FangGu, fracton, SFDM, hidden-sector).
7. `violationLevel_is_function` — uniqueness of `violationLevel`.
8. `ep_violation_is_vestigial_only` — central structural theorem (∀m violating WEP, m is vestigial).
9. `two_vestigial_mechanisms_distinct` — the two vestigial mechanisms are not the same object.
10. `distinct_violationLevel_implies_distinct_mechanism` — contrapositive.
11. `vestigial_vs_fangGu_distinct_EP_profile` — cross-mechanism tension.
12. `vestigial_microscope_violation_consistent` — bridge to `ClassificationTableDark.MicroscopeStatus`.

(Plus structural type definitions `EPLevel`, `EPMechanism`, `violationLevel`, `violatesAt`, `satisfiesAt` — counted as definitions, not theorems.)

**Strengthening 12 (Wave 6c.3 strengthening pass):**
13. `violatesAt_mono` — subsumption monotonicity.
14–17. `vestigialDifferentialCoupling_violationLevel`, `vestigialReliscSTEPClass_violationLevel`, `fangGuTorsionTrace_has_no_violation`, `fractonSubdiffusion_has_no_violation` — single-claim forms.
18–19. `sfdmThomasFermi_has_no_violation`, `hiddenSectorZ16Singlet_has_no_violation` — same.
20. `noViolation_implies_satisfiesAt` — bundle-extraction lemma.
21. `vestigial_phase_eta_violates_microscope_bound` — quantitative $1.0 > 10^{-15}$.
22. `vestigial_relics_below_microscope_bound` — quantitative $10^{-18} < 10^{-15}$.
23. `non_violators_share_violationLevel` — EP-degeneracy of four mechanisms.
24. `fangGu_failure_mode_is_kinematic_not_ep` — substantive cross-bridge to `FangGuTorsionDM.fg_cdm_obstruction`.

Note: a P5 self-equality `step_target_can_test_vestigial_relics : 1.0e-18 ≤ 1.0e-18 := le_refl _` and a `module_summary_marker : True := trivial` were REMOVED in the post-wave audit; the substantive content survives via `vestigial_relics_below_microscope_bound`.